# 08 — Agent Memory

This notebook demonstrates the orqest memory subsystem: storing, recalling, filtering, and forgetting knowledge across agent conversations.

**Use case:** A learning assistant that remembers facts about Python and uses them to enrich its answers.

**Prerequisites:**
- A `.env` file in the project root (or environment variables) with:
  ```
  LLM_API_KEY=your_api_key
  LLM_MODEL=openai:gpt-4.1
  ```

## 1. Setup

Load the orqest config and create a `LocalMemoryStore` backed by a temporary SQLite database. We use a temp directory so this notebook never pollutes your system.

In [ ]:
import tempfile
from pathlib import Path

from orqest import load_config
from orqest.memory import LocalMemoryStore, MemoryEntry, MemoryFilter

config = load_config()
print(f"Model:    {config.llm_model}")
print(f"API key:  {config.llm_api_key[:8]}...")

# Create a temp directory for the SQLite database
db_path = Path(tempfile.mkdtemp()) / "test_memory.db"
store = LocalMemoryStore(db_path)
print(f"DB path:  {db_path}")

## 2. Store memories

Let's store five facts about Python as semantic memories. Each fact comes from a different `source_agent`, simulating a multi-agent system where different specialists contribute knowledge.

In [ ]:
facts = [
    ("Python uses duck typing: if it walks like a duck and quacks like a duck, it is a duck.",
     "typing_expert"),
    ("Python 3.12 introduced type parameter syntax (PEP 695) for generic classes and functions.",
     "typing_expert"),
    ("The GIL (Global Interpreter Lock) prevents true parallel execution of Python bytecode in CPython.",
     "performance_agent"),
    ("Python list comprehensions are generally faster than equivalent for-loops due to optimized bytecode.",
     "performance_agent"),
    ("Python's asyncio module enables cooperative multitasking via coroutines and an event loop.",
     "concurrency_agent"),
]

stored_ids = []
for content, agent in facts:
    entry = MemoryEntry(
        content=content,
        memory_type="semantic",
        source_agent=agent,
        confidence=0.95,
    )
    entry_id = await store.store(entry)
    stored_ids.append(entry_id)
    print(f"Stored [{agent}]: {content[:60]}...")

print(f"\nTotal memories: {await store.count()}")

## 3. Recall memories

Query the store with a natural-language string. The `LocalMemoryStore` uses SQLite FTS5 full-text search to find matching entries.

In [ ]:
results = await store.recall("Python typing", k=5)

print(f"Found {len(results)} memories for query 'Python typing':\n")
for r in results:
    print(f"  [{r.source_agent}] (confidence={r.confidence}, reliability={r.reliability_score:.2f})")
    print(f"  {r.content}")
    print()

## 4. Filtered recall

Use `MemoryFilter` to narrow results by `source_agent` and `min_confidence`. This is useful when you only trust certain agents or want high-confidence facts.

In [ ]:
# Filter: only memories from the typing_expert with confidence >= 0.9
filters = MemoryFilter(source_agent="typing_expert", min_confidence=0.9)
filtered = await store.recall("Python", k=5, filters=filters)

print(f"Filtered results (source_agent='typing_expert', min_confidence=0.9):\n")
for r in filtered:
    print(f"  [{r.source_agent}] {r.content}")
    print()

# Filter: only from performance_agent
perf_filter = MemoryFilter(source_agent="performance_agent")
perf_results = await store.recall("Python", k=5, filters=perf_filter)

print(f"Filtered results (source_agent='performance_agent'):\n")
for r in perf_results:
    print(f"  [{r.source_agent}] {r.content}")

## 5. Self-healing: reliability decay

When a memory proves unreliable, calling `update_reliability(success=False)` multiplies its score by 0.7. After enough failures, the memory is automatically pruned (deleted when score falls below 0.1).

Let's store a "bad" memory and watch its reliability decay: 1.0 -> 0.7 -> 0.49 -> 0.343.

In [ ]:
# Store a dubious fact
bad_entry = MemoryEntry(
    content="Python is slower than JavaScript in every benchmark.",
    memory_type="semantic",
    source_agent="unreliable_agent",
    confidence=0.8,
)
bad_id = await store.store(bad_entry)
print(f"Stored bad memory: id={bad_id[:12]}..., reliability=1.0")

# Simulate three failures
for i in range(3):
    await store.update_reliability(bad_id, success=False)
    # Recall it to check the current score
    check = await store.recall("Python JavaScript", k=1)
    if check:
        score = check[0].reliability_score
        print(f"  After failure {i + 1}: reliability = {score:.3f}")
    else:
        print(f"  After failure {i + 1}: memory was pruned (score fell below 0.1)")

print(f"\nExpected decay: 1.0 x 0.7 = 0.700, x 0.7 = 0.490, x 0.7 = 0.343")

## 6. Agent with memory

Now let's build a real agent that recalls relevant memories before answering a question. We'll compare two responses: one from a plain agent (no memory) and one enriched with recalled facts.

In [ ]:
from pydantic import BaseModel, Field
from orqest.agents import BaseAgent, GlobalState
from orqest.memory import LocalMemoryStore


class QAOutput(BaseModel):
    """Structured answer from the QA agent."""
    answer: str = Field(description="A detailed answer to the user's question")
    sources_used: list[str] = Field(description="List of memory facts used, if any")


class QAAgent(BaseAgent[GlobalState, QAOutput]):
    """A QA agent that optionally enriches its prompt with recalled memories."""

    def __init__(self, model: str, api_key: str, memory_store: LocalMemoryStore | None = None):
        super().__init__(
            agent_name="qa_agent",
            system_prompt=(
                "You are a knowledgeable Python assistant. Answer the user's question "
                "accurately and concisely. If memory context is provided, incorporate "
                "those facts into your answer and list them in sources_used."
            ),
            output_type=QAOutput,
            model=model,
            api_key=api_key,
        )
        self._memory = memory_store

    async def _run_implementation(self, state: GlobalState, **kwargs) -> QAOutput:
        user_message = state.get_latest_message("user")
        prompt = str(user_message)

        # If we have a memory store, recall relevant facts and prepend them
        if self._memory is not None:
            memories = await self._memory.recall(str(user_message), k=3)
            if memories:
                context = "\n".join(f"- {m.content}" for m in memories)
                prompt = (
                    f"Relevant facts from memory:\n{context}\n\n"
                    f"User question: {user_message}"
                )

        result = await self.call_model(prompt, state)
        return result.output

In [ ]:
question = "How does Python handle typing and what changed in Python 3.12?"

# --- Without memory ---
agent_no_memory = QAAgent(model=config.llm_model, api_key=config.llm_api_key)
state_plain = GlobalState()
state_plain.add_message("user", question)
output_plain = await agent_no_memory.run(state_plain)

print("=== Without memory ===")
print(f"Answer: {output_plain.answer[:300]}...")
print(f"Sources used: {output_plain.sources_used}")

# --- With memory ---
agent_with_memory = QAAgent(model=config.llm_model, api_key=config.llm_api_key, memory_store=store)
state_memory = GlobalState()
state_memory.add_message("user", question)
output_memory = await agent_with_memory.run(state_memory)

print("\n=== With memory ===")
print(f"Answer: {output_memory.answer[:300]}...")
print(f"Sources used: {output_memory.sources_used}")

## 7. Forget

Remove a specific memory by its id and verify it's gone.

In [ ]:
target_id = stored_ids[0]
print(f"Forgetting memory: {target_id[:12]}...")
print(f"Memories before forget: {await store.count()}")

await store.forget(target_id)

print(f"Memories after forget:  {await store.count()}")

# Verify it's gone by trying to recall its content
check = await store.recall("duck typing", k=5)
found_ids = [r.id for r in check]
if target_id in found_ids:
    print("Memory still found (unexpected)")
else:
    print("Memory successfully forgotten")

## 8. Assessment

Quick summary of memory store state and recall latency.

In [ ]:
import time

total = await store.count()
print(f"Total memories in store: {total}")

# Measure recall latency
start = time.perf_counter()
_ = await store.recall("Python performance", k=5)
elapsed_ms = (time.perf_counter() - start) * 1000
print(f"Recall latency: {elapsed_ms:.2f} ms")

# Show all memories with their reliability scores
all_memories = await store.recall("Python", k=10)
print(f"\nAll retrievable memories ({len(all_memories)}):")
for m in all_memories:
    print(f"  [{m.source_agent:20s}] reliability={m.reliability_score:.3f}  {m.content[:70]}...")

## What's happening under the hood

1. **SQLite + FTS5**: `LocalMemoryStore` creates a SQLite database with a `memories` table and an FTS5 virtual table (`memories_fts`) for full-text search. Triggers keep the FTS index in sync with inserts, updates, and deletes. If FTS5 is unavailable (rare), the store falls back to `LIKE` queries.

2. **Reliability decay math**: Each call to `update_reliability(success=False)` multiplies the score by 0.7:
   - Start: `1.0`
   - Failure 1: `1.0 x 0.7 = 0.700`
   - Failure 2: `0.7 x 0.7 = 0.490`
   - Failure 3: `0.49 x 0.7 = 0.343`
   - After ~7 failures the score drops below 0.1 and the entry is automatically pruned (deleted).

3. **Access tracking**: Every `recall()` updates `last_accessed` and increments `access_count` on matched entries, giving you recency and frequency signals for future retrieval strategies.

4. **Best-effort operations**: All store operations are wrapped in try/except. Errors are logged via loguru but never raised to the caller, so a database hiccup cannot crash your agent pipeline.

5. **MemoryFilter**: Filters are applied as SQL WHERE clauses — `source_agent`, `memory_type`, `min_confidence`, and `min_reliability` map directly to column comparisons, keeping queries efficient.

## Cleanup

Close the database connection and remove the temp directory.

In [ ]:
import shutil

await store.close()
shutil.rmtree(db_path.parent, ignore_errors=True)
print("Cleaned up temp database.")